# Elastic Net Regression

## What is Elastic Net?

Elastic Net is a regularized Linear Regression model that combines:

```text id="thzj6e"
RIDGE                 LASSO
 L2                     L1
  \                     /
   \                   /
    └── ELASTIC NET ──┘
```

It gets benefits from both **Ridge and Lasso regularization**.

---

## Why Elastic Net?

We learned:

### Ridge

```text id="atvdsq"
L2 Regularization
       ↓
Shrinks coefficients
       ↓
Generally keeps all features
       ↓
Works well with correlated features
```

### Lasso

```text id="vfh30j"
L1 Regularization
       ↓
Shrinks coefficients
       ↓
Some coefficients → 0
       ↓
Feature Selection
```

Elastic Net combines both ideas.

```text id="7hrygu"
          ELASTIC NET
               │
        ┌──────┴──────┐
        ↓             ↓
       L1             L2
        ↓             ↓
Feature Selection   Coefficient
                    Stabilization
```

---

# Elastic Net Loss

Conceptually:

[
Loss =
MSE
+
L1\ Penalty
+
L2\ Penalty
]

Elastic Net therefore controls both:

* Overall regularization strength
* Balance between Lasso and Ridge

---

# The Two Important Controls

## 1. `alpha`

Controls the **overall strength of regularization**.

```text id="ry5p6i"
alpha ↑
   ↓
Regularization ↑
   ↓
Coefficients become more constrained
   ↓
Too large
   ↓
UNDERFITTING
```

---

## 2. `l1_ratio`

Controls the balance between **Lasso and Ridge**.

```text id="1vd0af"
l1_ratio = 0
      ↓
Ridge-like


l1_ratio = 0.5
      ↓
Mix of Lasso + Ridge


l1_ratio = 1
      ↓
Lasso
```

### Easy Memory

```text id="9pl12k"
0 -------------------------- 1

RIDGE        MIX          LASSO
 L2                        L1
```

---

# Example

```python id="e6mjg4"
ElasticNet(
    alpha=0.1,
    l1_ratio=0.5
)
```

Means:

```text id="4y6s0i"
alpha = 0.1
→ overall regularization strength

l1_ratio = 0.5
→ balanced mixture of L1 and L2
```

---

# Why Use Elastic Net?

Elastic Net is useful when:

* There are many features
* Some features may be unnecessary
* Features are correlated
* You want coefficient shrinkage
* You also want feature selection
* Pure Lasso selection may be unstable

---

# Ridge vs Lasso vs Elastic Net

| Model       | Regularization | Feature Selection | Correlated Features |
| ----------- | -------------- | ----------------- | ------------------- |
| Ridge       | L2             | ❌ Generally no    | ✅ Good              |
| Lasso       | L1             | ✅ Yes             | Can be unstable     |
| Elastic Net | L1 + L2        | ✅ Yes             | ✅ Often useful      |

---

# 🧠 Quick Recall Sketch

```text id="trmq8b"
                LINEAR REGRESSION
                       │
                       ▼
                REGULARIZATION
                       │
          ┌────────────┴────────────┐
          │                         │
          ▼                         ▼
        RIDGE                     LASSO
         L2                         L1
          │                         │
          ▼                         ▼
       SHRINK                    SHRINK
    coefficients              coefficients
          │                         +
          │                       SELECT
          │                         │
          └──────────┬──────────────┘
                     ▼
                ELASTIC NET
                     │
                  L1 + L2
                     │
          ┌──────────┴──────────┐
          ▼                     ▼
       SHRINK                 SELECT
    coefficients             features
```

---

# 🎚️ Two Knobs to Remember

```text id="3v6rgq"
ELASTIC NET

KNOB 1
──────
alpha

"How much regularization?"


KNOB 2
──────
l1_ratio

"What mixture?"

0 ←──────────────→ 1

Ridge             Lasso
```

---

# Key Interview Answer

> **Elastic Net combines L1 and L2 regularization. It can perform feature selection like Lasso while also benefiting from Ridge-style coefficient stabilization, making it useful when there are many features and correlated predictors.**

---

# 10-Second Recall

```text id="e7g4h1"
Ridge
= L2
= Shrink


Lasso
= L1
= Shrink + Select


Elastic Net
= L1 + L2
= Shrink + Select + Balance
```

## One-Line Memory

> **Elastic Net = Ridge + Lasso**


In [1]:

from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)

X = housing.data
y = housing.target

In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline

en_model = Pipeline([
    ("scalar",StandardScaler()),
    ("model",ElasticNet(alpha=0.01,l1_ratio=0.5))
])

In [15]:
en_model.fit(X_train,y_train)

Pipeline(steps=[('scalar', StandardScaler()),
                ('model', ElasticNet(alpha=0.01))])

In [16]:
y_train_pred = en_model.predict(X_train)
y_test_pred = en_model.predict(X_test)

In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate(model, X, y):
    pred = model.predict(X)

    return {
        "R2": round(r2_score(y, pred), 3),
        "MAE": round(mean_absolute_error(y, pred), 3),
        "RMSE": round(np.sqrt(mean_squared_error(y, pred)), 3)
    }

In [18]:
print("TEST")
print(evaluate(en_model, X_test, y_test))
print("TRAIN")
print(evaluate(en_model, X_train, y_train))

TEST
{'R2': 0.58, 'MAE': 0.534, 'RMSE': 0.742}
TRAIN
{'R2': 0.61, 'MAE': 0.53, 'RMSE': 0.722}


In [19]:
alphas = [0.001, 0.01, 0.1, 1]
ratios = [0.2, 0.5, 0.8]

for alpha in alphas:
    for ratio in ratios:

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("model", ElasticNet(
                alpha=alpha,
                l1_ratio=ratio,
                max_iter=10000
            ))
        ])

        model.fit(X_train, y_train)

        coef = model.named_steps["model"].coef_

        print(
            f"Alpha: {alpha} | "
            f"L1 Ratio: {ratio} | "
            f"Train R²: {model.score(X_train, y_train):.3f} | "
            f"Test R²: {model.score(X_test, y_test):.3f} | "
            f"Selected: {(coef != 0).sum()}"
        )
        

Alpha: 0.001 | L1 Ratio: 0.2 | Train R²: 0.613 | Test R²: 0.576 | Selected: 8
Alpha: 0.001 | L1 Ratio: 0.5 | Train R²: 0.613 | Test R²: 0.577 | Selected: 8
Alpha: 0.001 | L1 Ratio: 0.8 | Train R²: 0.613 | Test R²: 0.577 | Selected: 8
Alpha: 0.01 | L1 Ratio: 0.2 | Train R²: 0.611 | Test R²: 0.579 | Selected: 7
Alpha: 0.01 | L1 Ratio: 0.5 | Train R²: 0.610 | Test R²: 0.580 | Selected: 7
Alpha: 0.01 | L1 Ratio: 0.8 | Train R²: 0.609 | Test R²: 0.581 | Selected: 7
Alpha: 0.1 | L1 Ratio: 0.2 | Train R²: 0.558 | Test R²: 0.541 | Selected: 6
Alpha: 0.1 | L1 Ratio: 0.5 | Train R²: 0.531 | Test R²: 0.515 | Selected: 4
Alpha: 0.1 | L1 Ratio: 0.8 | Train R²: 0.503 | Test R²: 0.487 | Selected: 3
Alpha: 1 | L1 Ratio: 0.2 | Train R²: 0.315 | Test R²: 0.306 | Selected: 1
Alpha: 1 | L1 Ratio: 0.5 | Train R²: 0.208 | Test R²: 0.203 | Selected: 1
Alpha: 1 | L1 Ratio: 0.8 | Train R²: 0.000 | Test R²: -0.000 | Selected: 0
